# Candidate SSE Socio-Geodemographic Association: Window Sensitivity

Runs the clade-grouped composition and node-mixing regression pipeline using window-level surveillance adjusters instead of `C(window_idx)`: proportion sequenced and positive tests. Clade remains excluded from the adjustment set because models are fitted within clade groups.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "config.yaml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from sse_detection.lib.association_pipeline import (  # noqa: E402
    default_model_sets,
    run_association_pipeline,
)

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 220)

## Model Specification

In [2]:
RESULT_DIR = PROJECT_ROOT / "sse_detection" / "window_sensitivity"
MODEL_METHOD = "firth_glm"

MODEL_SETS = default_model_sets(
    variant_adjuster=None,
    window_adjustment="surveillance",
)

MODEL_SETS

{'primary': ['z_wn_prop_sequenced', 'z_log1p_wn_positive_tests'],
 'expanded': ['z_wn_prop_sequenced',
  'z_log1p_wn_positive_tests',
  'z_dz_cum_prop_sequenced',
  'z_dz_cum_incidence_per_capita',
  'z_dz_7d_test_positivity',
  'z_log1p_dz_cum_positive_tests']}

## Fit and Save

In [3]:
result = run_association_pipeline(
    project_root=PROJECT_ROOT,
    result_dir=RESULT_DIR,
    model_method=MODEL_METHOD,
    variant_adjuster=None,
    window_adjustment="surveillance",
    composition_model_sets=MODEL_SETS,
    mixing_model_sets=MODEL_SETS,
    group_by_clade=True,
)

print(f"Results saved to: {result['result_dir']}")
display(result["cluster_diagnostics"])
{name: len(table) for name, table in result["summary_tables"].items()}

Fitted composition__primary__single__sex__20b: 1,424 rows
Fitted composition__primary__single__age_band__20b: 1,424 rows
Fitted composition__primary__single__simd_quintile__20b: 1,424 rows
Fitted composition__primary__single__urban_rural_class__20b: 1,424 rows
Fitted composition__primary__single__health_board__20b: 1,424 rows
Fitted composition__primary__joint__20b: 1,424 rows
Fitted composition__primary__single__sex__20a: 687 rows
Fitted composition__primary__single__age_band__20a: 687 rows
Fitted composition__primary__single__simd_quintile__20a: 687 rows
Fitted composition__primary__single__urban_rural_class__20a: 687 rows
Fitted composition__primary__single__health_board__20a: 687 rows
Fitted composition__primary__joint__20a: 687 rows
Fitted composition__primary__single__sex__20e_eu1: 6,443 rows
Fitted composition__primary__single__age_band__20e_eu1: 6,443 rows
Fitted composition__primary__single__simd_quintile__20e_eu1: 6,443 rows
Fitted composition__primary__single__urban_rural_cl

,cluster_col,n_rows,n_clusters,min_rows_per_cluster,median_rows_per_cluster,outcome_positive_clusters,outcome_varying_clusters,analysis_frame
0,cluster_id,264139,13059,1,9.0,6907,0,composition
1,cluster_id,13059,13059,1,1.0,6907,0,node_mixing


{'composition_wald.csv': 260,
 'composition_odds_ratios.csv': 1924,
 'composition_fit_stats.csv': 156,
 'mixing_wald.csv': 260,
 'mixing_odds_ratios.csv': 260,
 'mixing_fit_stats.csv': 156}

In [4]:
if not result["failures"].empty:
    display(result["failures"])
else:
    print("No model failures recorded.")

No model failures recorded.
